# GazeSpeakZero — E3: from the objects a person picks to the sentence they mean

**Before you start: Runtime → Change runtime type → T4 GPU → Save.**

Run the cells **one at a time** with Shift + Enter. About 1.5 hours in total, most of it unattended. Everything is saved to Google Drive (`MyDrive/gazespeakzero/E3`), so if Colab disconnects, run the cells again and it carries on where it stopped.

What it does: 60 messages a person with severe motor impairment might want to say, each with the objects they would select. Every method has to recover the message from the objects, under four conditions: correct objects, only the first object, one object wrong because the scene model mislabelled it (drawn from E2's real errors), and one object wrong because of a gaze mis-selection.

Methods: a fixed template, a frequency rule, sentence-embedding retrieval, and four small open language models used two ways (ranking known messages, and writing the message themselves).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/gazespeakzero/E3'
E2 = '/content/drive/MyDrive/gazespeakzero/E2'
import os, torch
os.makedirs(WORK, exist_ok=True)
print(WORK)
print('E2 items found:', os.path.exists(f'{E2}/results_v2/e2_items.csv') or os.path.exists(f'{E2}/results/e2_items.csv'))
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch the runtime to T4 GPU before continuing')

In [ ]:
!pip -q install "transformers>=4.51,<5" "sentence-transformers>=3.0" "accelerate>=0.30" pandas tabulate
import transformers, sentence_transformers
print('transformers', transformers.__version__, '| sentence-transformers', sentence_transformers.__version__)

In [ ]:
!mkdir -p /content/e3 /content/e2

In [ ]:
%%writefile /content/e2/aac_vocab.py
"""
E2: the AAC-relevant vocabulary and the text-to-LVIS matcher. Fixed before any
E2 result was seen.

AAC_VOCAB lists everyday objects that a person with severe motor impairment at
home or in a care setting might ask for, use or talk about: drink and food,
personal care and medical items, comfort and bedding, devices, furniture, and
pets. The entries are LVIS category names. Entries that the LVIS version in use
does not contain are reported and dropped, never silently remapped.
"""
from __future__ import annotations

import re

AAC_VOCAB = [
    # drink and food
    "bottle", "water_bottle", "cup", "mug", "glass_(drink_container)", "wineglass", "kettle", "teapot",
    "coffee_maker", "milk", "orange_juice", "banana", "apple", "orange_(fruit)", "bread", "sandwich",
    "cake", "cookie", "doughnut", "pizza", "bowl", "plate", "spoon", "fork", "knife", "straw_(for_drinking)",
    "napkin",
    # personal care and medical
    "toothbrush", "toothpaste", "hairbrush", "comb", "towel", "hand_towel", "soap", "toilet_tissue",
    "tissue_paper", "medicine", "pill", "spectacles", "wheelchair", "crutch", "walking_stick",
    "toilet", "sink", "bathtub", "shower_head",
    # comfort, bedding, clothing
    "bed", "pillow", "cushion", "blanket", "quilt", "bedspread", "slipper_(footwear)", "sock", "shoe",
    "jacket", "sweater", "hat",
    # environment and devices
    "lamp", "table_lamp", "fan", "heater", "air_conditioner", "curtain", "clock", "wall_clock",
    "alarm_clock", "television_set", "remote_control", "radio_receiver", "cellular_telephone",
    "telephone", "laptop_computer", "book", "newspaper", "magazine", "pen", "notebook", "headset",
    "earphone",
    # furniture
    "chair", "armchair", "recliner", "sofa", "table", "dining_table", "coffee_table", "desk",
    "refrigerator", "microwave_oven", "stove", "trash_can",
    # other
    "dog", "cat", "vase", "painting", "mirror", "flower_arrangement",
]

ROOM_INDICATORS = [          # priority order: the first matching room wins
    ("bathroom", ["toilet", "bathtub", "shower_curtain", "shower_head"]),
    ("bedroom", ["bed", "bunk_bed"]),
    ("kitchen", ["refrigerator", "stove", "microwave_oven", "oven", "kitchen_sink", "dishwasher"]),
    ("living room", ["sofa", "armchair", "television_set", "coffee_table"]),
    ("dining", ["dining_table"]),
]

STOP = {"a", "an", "the", "some", "of", "with", "on", "in", "and", "or", "for", "to", "at", "by"}
CUT = {"of", "for", "on", "with", "in", "at", "from", "near"}   # 'glass of water' -> 'glass'

# Everyday words -> LVIS names, fixed before any output was seen. Applied to every
# method alike. A target missing from the LVIS release in use is skipped.
ALIASES = {
    "fridge": "refrigerator", "laptop": "laptop_computer", "microwave": "microwave_oven",
    "tv": "television_set", "television": "television_set", "telly": "television_set",
    "remote": "remote_control", "tv remote": "remote_control",
    "phone": "cellular_telephone", "cell phone": "cellular_telephone", "cellphone": "cellular_telephone",
    "mobile phone": "cellular_telephone", "mobile": "cellular_telephone", "smartphone": "cellular_telephone",
    "wine glass": "wineglass", "glasses": "spectacles", "eyeglasses": "spectacles",
    "couch": "sofa", "settee": "sofa", "bin": "trash_can", "dustbin": "trash_can", "garbage can": "trash_can",
    "toilet paper": "toilet_tissue", "tissues": "tissue_paper", "tissue": "tissue_paper",
    "water glass": "glass_(drink_container)", "drinking glass": "glass_(drink_container)",
    "mouse": "mouse_(computer_equipment)", "computer mouse": "mouse_(computer_equipment)",
    "tap": "faucet", "duvet": "quilt", "comforter": "quilt", "bedsheet": "sheet_(bedding)",
    "potted plant": "flowerpot", "plant pot": "flowerpot", "teddy": "teddy_bear",
}

# Lenient scoring: an item also counts as correct if a category in the same group is present.
LENIENT_GROUPS = [
    {"table", "dining_table", "coffee_table", "kitchen_table", "desk"},
    {"clock", "wall_clock", "alarm_clock"},
    {"lamp", "table_lamp"},
    {"towel", "hand_towel", "bath_towel", "dishtowel", "paper_towel"},
    {"blanket", "quilt", "bedspread"},
    {"cup", "mug"},
    {"glass_(drink_container)", "wineglass"},
    {"telephone", "cellular_telephone"},
    {"bottle", "water_bottle", "beer_bottle", "wine_bottle"},
    {"pillow", "cushion"},
]


def canon(name: str) -> str:
    """LVIS name -> plain words: 'glass_(drink_container)' -> 'glass', 'television_set' -> 'television set'."""
    name = re.sub(r"_?\(.*?\)", "", name)
    return name.replace("_", " ").strip().lower()


def _singular_candidates(w: str) -> list[str]:
    out = []
    for suf, rep in (("ies", "y"), ("ves", "f"), ("ves", "fe"), ("oes", "o"), ("ches", "ch"),
                     ("shes", "sh"), ("sses", "ss"), ("xes", "x"), ("es", ""), ("s", "")):
        if w.endswith(suf) and len(w) > len(suf) + 1:
            out.append(w[: -len(suf)] + rep)
    return out


class Matcher:
    """Maps free text ('Two white pillows') to one LVIS category id, or None.

    Tries, in order: the whole phrase, the phrase singularised, then shorter
    trailing phrases (so 'wooden coffee table' -> 'coffee table' -> 'table'),
    always by exact equality with a canonicalised name or synonym."""

    def __init__(self, categories: list[dict]):
        self.id2name = {c["id"]: c["name"] for c in categories}
        by_name = {c["name"]: c["id"] for c in categories}
        self.lookup: dict[str, int] = {}
        self.clashes: dict[str, list[str]] = {}
        for c in sorted(categories, key=lambda c: c["id"]):
            for s in [c["name"]] + list(c.get("synonyms", [])):
                k = canon(s)
                if k in self.lookup and self.lookup[k] != c["id"]:
                    self.clashes.setdefault(k, [self.id2name[self.lookup[k]]]).append(c["name"])
                self.lookup.setdefault(k, c["id"])          # first (lowest id) wins on a clash
        own = {}                                            # a category's own name beats others' synonyms
        for c in sorted(categories, key=lambda c: c["id"]):
            own.setdefault(canon(c["name"]), c["id"])       # name-vs-name clash: lowest id
        self.lookup.update(own)
        self.aliases_used = {}
        for k, target in ALIASES.items():
            if target in by_name:
                self.lookup[k] = by_name[target]
                self.aliases_used[k] = target
        self.group = {}
        for g in LENIENT_GROUPS:
            ids = {by_name[n] for n in g if n in by_name}
            for i in ids:
                self.group[i] = ids
        self.head = {cid: canon(n).split()[-1] for cid, n in self.id2name.items()}

    def match(self, text: str):
        t = re.sub(r"[^a-z\s-]", " ", text.lower()).replace("-", " ")
        raw = t.split()
        whole = " ".join(w for w in raw if w not in {"a", "an", "the"})
        if whole in self.lookup:                         # 'chest of drawers' as a whole name
            return self.lookup[whole]
        for j, w in enumerate(raw):                      # 'glass of water' -> 'glass'
            if w in CUT and j > 0:
                raw = raw[:j]
                break
        words = [w for w in raw if w not in STOP]
        if not words:
            return None
        for i in range(len(words)):                      # longest trailing phrase first
            phrase = words[i:]
            cands = [" ".join(phrase)] + [" ".join(phrase[:-1] + [w]) for w in _singular_candidates(phrase[-1])]
            for cand in cands:
                if cand in self.lookup:
                    return self.lookup[cand]
        return None

    def resolve_vocab(self, names: list[str]):
        ids, missing = [], []
        by_name = {v: k for k, v in self.id2name.items()}
        for n in names:
            cid = by_name.get(n)                               # exact LVIS names only
            if cid is None:
                missing.append(n)
            elif cid not in ids:
                ids.append(cid)
        return ids, missing


In [ ]:
%%writefile /content/e3/build_intent_bank.py
"""
Build and validate e3/intent_bank.json.

Every object must exist in the E2 AAC vocabulary (aac_vocab.py), so that E2's
output and E3's input are the same vocabulary. Run:

    python build_intent_bank.py            # writes intent_bank.json and prints checks
"""
import json
import os
import sys

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, os.path.join(HERE, "..", "e2"))
from aac_vocab import AAC_VOCAB, canon  # noqa: E402

# purpose: needs | information | social | etiquette   (Light 1988)
# source: the published category the intent comes from (see E3_DESIGN.md)
B = [
    # ---------------- wants and needs (33) ----------------
    ("bedroom", "needs", "board:thirst", ["water_bottle", "straw_(for_drinking)"],
     "I am thirsty, can I have some water?",
     ["Could I have a drink of water, please?", "I would like water with a straw."]),
    ("bedroom", "needs", "board:medication", ["pill", "water_bottle"],
     "It is time for my medicine, please bring it with water.",
     ["Can I have my tablet with some water?", "My pills are due now."]),
    ("bedroom", "needs", "board:temperature", ["blanket"],
     "I am cold, can I have another blanket?",
     ["Please put another blanket over me.", "I feel cold, I need a blanket."]),
    ("bedroom", "needs", "board:temperature", ["fan"],
     "I am too warm, please turn on the fan.",
     ["Could you switch the fan on? It is hot.", "Please put the fan on for me."]),
    ("bedroom", "needs", "board:position", ["pillow"],
     "Please move my pillow, my head is uncomfortable.",
     ["Can you fix my pillow?", "My pillow needs adjusting."]),
    ("bedroom", "needs", "board:position", ["bed", "quilt", "pillow"],
     "Please straighten my bed covers.",
     ["The quilt needs pulling up.", "Could you tidy my bedding?"]),
    ("bedroom", "needs", "board:toileting", ["toilet"],
     "I need to use the toilet.",
     ["Please take me to the toilet.", "I have to go to the bathroom."]),
    ("bedroom", "needs", "board:secretions", ["tissue_paper"],
     "I need a tissue, please.",
     ["Could you pass me a tissue?", "Please wipe my mouth with a tissue."]),
    ("bedroom", "needs", "board:sleep", ["lamp"],
     "Please turn the light off, I want to sleep.",
     ["Switch the lamp off, I am tired.", "Turn off the bedside light, please."]),
    ("bedroom", "needs", "board:vision", ["spectacles"],
     "Can you pass me my glasses?",
     ["I need my spectacles.", "Please put my glasses on for me."]),
    ("bedroom", "needs", "board:mobility", ["walking_stick"],
     "Pass me my walking stick, I want to move.",
     ["I need my stick.", "Bring me my walking cane, please."]),
    ("bedroom", "needs", "board:dressing", ["sock", "slipper_(footwear)", "shoe"],
     "Please put my socks and slippers on.",
     ["I want my socks and slippers.", "Could you dress my feet?"]),
    ("bathroom", "needs", "board:hygiene", ["toothbrush", "toothpaste", "cup"],
     "I want to brush my teeth.",
     ["Please help me clean my teeth.", "Bring my toothbrush and toothpaste."]),
    ("bathroom", "needs", "board:hygiene", ["shower_head", "towel", "soap"],
     "I would like a shower now.",
     ["Please help me wash in the shower.", "I want to be showered."]),
    ("bathroom", "needs", "board:toileting", ["toilet_tissue"],
     "The toilet paper is finished, please bring more.",
     ["We need more toilet roll.", "There is no toilet paper left."]),
    ("bathroom", "needs", "board:hygiene", ["soap", "sink"],
     "I want to wash my hands.",
     ["Please help me wash my hands with soap.", "Can I clean my hands at the sink?"]),
    ("bathroom", "needs", "board:grooming", ["hairbrush"],
     "Please brush my hair.",
     ["Could you do my hair?", "I would like my hair brushed."]),
    ("bathroom", "needs", "board:grooming", ["mirror"],
     "Hold the mirror up, I want to see myself.",
     ["Let me look in the mirror.", "Please show me my face in the mirror."]),
    ("kitchen", "needs", "board:thirst", ["kettle", "mug"],
     "I would like a hot drink, please.",
     ["Could you make me a cup of tea?", "Please boil the kettle for me."]),
    ("kitchen", "needs", "board:thirst", ["milk", "glass_(drink_container)"],
     "Can I have a glass of milk?",
     ["I would like some milk.", "Please pour me milk in a glass."]),
    ("kitchen", "needs", "board:hunger", ["sandwich"],
     "I am hungry, I would like a sandwich.",
     ["Could you make me a sandwich?", "I want something to eat, a sandwich."]),
    ("kitchen", "needs", "board:thirst", ["refrigerator", "orange_juice"],
     "There is juice in the fridge, can I have some?",
     ["Please get the orange juice from the fridge.", "I would like juice, it is in the refrigerator."]),
    ("kitchen", "needs", "board:medication", ["medicine", "spoon"],
     "My liquid medicine is due, please give it to me.",
     ["Bring my medicine and a spoon.", "It is time for my syrup."]),
    ("dining", "needs", "board:hunger", ["plate", "fork", "napkin"],
     "I am ready to eat now.",
     ["Please bring my meal.", "I would like my food."]),
    ("dining", "needs", "board:thirst", ["cup", "straw_(for_drinking)"],
     "Can I have a drink with a straw?",
     ["Please give me my cup with a straw.", "I need a straw for my drink."]),
    ("dining", "needs", "board:hygiene", ["napkin"],
     "I need a napkin, please.",
     ["Could you pass me a serviette?", "Please wipe my mouth with a napkin."]),
    ("dining", "needs", "board:feeding", ["spoon"],
     "Please feed me with a spoon, it is easier.",
     ["Use the spoon to feed me.", "I would rather be fed with a spoon."]),
    ("living room", "needs", "board:entertainment", ["television_set", "remote_control"],
     "Please turn on the television.",
     ["Switch the TV on for me.", "I want to watch television."]),
    ("living room", "needs", "board:position", ["sofa", "cushion"],
     "I want to sit on the sofa.",
     ["Please move me to the couch.", "Help me onto the sofa with a cushion."]),
    ("living room", "needs", "board:mobility", ["wheelchair"],
     "I want to get into my wheelchair.",
     ["Please transfer me to the wheelchair.", "Put me in my chair with wheels."]),
    ("living room", "needs", "board:light", ["curtain"],
     "Please open the curtains.",
     ["Could you draw the curtains back?", "I want to see outside, open the curtain."]),
    ("living room", "needs", "board:temperature", ["heater"],
     "Please turn up the heater.",
     ["It is cold, put the heating on.", "Could you increase the heater?"]),
    ("bedroom", "needs", "board:comfort", ["hand_towel"],
     "My face is sweaty, please wipe it with a towel.",
     ["Could you dab my forehead with a towel?", "Please use a towel on my face."]),
    # ---------------- information exchange (10) ----------------
    ("bedroom", "information", "info:time", ["alarm_clock"],
     "What time is it?",
     ["Could you tell me the time?", "Is it morning already?"]),
    ("living room", "information", "info:schedule", ["wall_clock"],
     "When is my appointment today?",
     ["What time do we leave for the clinic?", "Tell me the time of my appointment."]),
    ("bedroom", "information", "info:medical", ["medicine"],
     "I have already taken my medicine.",
     ["My tablets are done for now.", "Do not give me the medicine again."]),
    ("bedroom", "information", "info:instruction", ["notebook", "pen"],
     "Write this down for me, please.",
     ["Take a note of this.", "Please use the notebook to write it."]),
    ("living room", "information", "info:instruction", ["laptop_computer"],
     "Open my laptop, I want to check my email.",
     ["Please start the computer for me.", "I need my laptop to read messages."]),
    ("bedroom", "information", "info:entertainment", ["book"],
     "Read me the next chapter of my book.",
     ["Please continue my book.", "Read to me from that book."]),
    ("living room", "information", "info:news", ["newspaper"],
     "Read me the news, please.",
     ["What does the paper say today?", "Tell me the headlines from the newspaper."]),
    ("kitchen", "information", "info:safety", ["stove"],
     "Be careful, the stove is still on.",
     ["Turn the cooker off, it is burning.", "The gas is still lit."]),
    ("bedroom", "information", "info:maintenance", ["table_lamp"],
     "The lamp beside my bed is not working.",
     ["My bedside light is broken.", "That lamp needs a new bulb."]),
    ("dining", "information", "info:household", ["cake"],
     "There is cake for the visitors, offer them some.",
     ["Give our guests some cake.", "The cake is for everybody."]),
    # ---------------- social closeness (10) ----------------
    ("bedroom", "social", "social:family", ["cellular_telephone"],
     "Call my family, I want to talk to them.",
     ["Please phone my daughter.", "I would like to speak to my family on the phone."]),
    ("living room", "social", "social:conversation", ["painting"],
     "That painting is my favourite, tell me about it.",
     ["I like looking at that picture.", "Talk to me about the painting."]),
    ("living room", "social", "social:pets", ["dog"],
     "Bring the dog to me, I want to stroke it.",
     ["Let the dog come up here.", "I want to pet the dog."]),
    ("bedroom", "social", "social:pets", ["cat"],
     "Is the cat here? I would like to see her.",
     ["Bring the cat in, please.", "Where is my cat?"]),
    ("living room", "social", "social:conversation", ["flower_arrangement"],
     "Who sent these flowers?",
     ["Tell me who brought the flowers.", "Where did that bouquet come from?"]),
    ("dining", "social", "social:celebration", ["wineglass"],
     "Pour me a little wine, let us celebrate.",
     ["A small glass of wine for me too.", "I would like to join the toast."]),
    ("living room", "social", "social:music", ["radio_receiver"],
     "Put some music on, please.",
     ["Turn the radio on.", "I would like to listen to music."]),
    ("bedroom", "social", "social:conversation", ["magazine"],
     "Sit with me and look at this magazine.",
     ["Let us go through the magazine together.", "Stay and read the magazine with me."]),
    ("living room", "social", "social:company", ["armchair"],
     "Sit down next to me for a while.",
     ["Take the armchair and stay with me.", "Please keep me company here."]),
    ("bedroom", "social", "social:music", ["headset"],
     "Put my headphones on, I want to listen.",
     ["Help me with my headset.", "I would like my headphones."]),
    # ---------------- social etiquette (7) ----------------
    ("dining", "etiquette", "etiquette:thanks", ["plate"],
     "That was lovely, thank you.",
     ["The meal was very good, thank you.", "Thank you for my food."]),
    ("living room", "etiquette", "etiquette:offer", ["cup", "teapot", "milk"],
     "Would you like a cup of tea?",
     ["Have some tea with me.", "Shall we offer our guest tea?"]),
    ("dining", "etiquette", "etiquette:welcome", ["chair"],
     "Please sit down, make yourself comfortable.",
     ["Take a seat, please.", "Do sit down with us."]),
    ("bedroom", "etiquette", "etiquette:thanks", ["flower_arrangement"],
     "Thank you for the flowers, they are beautiful.",
     ["It was kind of you to bring flowers.", "The flowers are lovely, thank you."]),
    ("living room", "etiquette", "etiquette:apology", ["television_set"],
     "Sorry, could you turn the television down?",
     ["The TV is a little loud, sorry.", "Please lower the television volume."]),
    ("dining", "etiquette", "etiquette:apology", ["napkin", "table"],
     "Excuse me, I have spilled something on the table.",
     ["Sorry, I have made a mess here.", "I need help, I spilt my drink."]),
    ("bedroom", "etiquette", "etiquette:welcome", ["chair", "bed"],
     "Please come in and sit down beside me.",
     ["Bring a chair over and sit with me.", "You are welcome, sit here."]),
]


def main():
    bank, bad = [], []
    for i, (room, purpose, source, objs, utt, paras) in enumerate(B, 1):
        for o in objs:
            if o not in AAC_VOCAB:
                bad.append((i, o))
        bank.append({"id": f"I{i:02d}", "room": room, "purpose": purpose, "source": source,
                     "objects": [canon(o) for o in objs], "objects_raw": objs,
                     "utterance": utt, "paraphrases": paras})
    if bad:
        raise SystemExit(f"objects not in the AAC vocabulary: {bad}")
    out = os.path.join(HERE, "intent_bank.json")
    json.dump(bank, open(out, "w"), indent=1)

    from collections import Counter
    first = Counter(b["objects"][0] for b in bank)
    shared = sum(v for v in first.values() if v > 1)
    print(f"{len(bank)} intents -> {out}")
    print("purposes:", dict(Counter(b['purpose'] for b in bank)))
    print("rooms:", dict(Counter(b['room'] for b in bank)))
    print("object-list lengths:", dict(Counter(len(b['objects']) for b in bank)))
    print(f"intents sharing their first object with another intent: {shared}")
    print("duplicate utterances:", [u for u, c in Counter(b['utterance'] for b in bank).items() if c > 1])
    assert len({b["utterance"] for b in bank}) == len(bank), "duplicate utterance"


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/e3/e3_build_cases.py
"""
E3, step 1: turn the intent bank into the cases the models see, under four conditions.

    python e3_build_cases.py --bank intent_bank.json --e2-items ../e2/results/e2_items.csv --out cases.jsonl

Conditions
----------
clean        the intent's own objects
partial      the first object only
wrong_label  one object replaced by an item E2 actually proposed for that room and the
             labels verified as ABSENT (a real labelling error, not an invented one)
wrong_pick   one object replaced by another item that appeared on the same first screen
             of three in E2 for that room (a plausible gaze mis-selection)

Everything is drawn with a fixed seed, so every model sees exactly the same cases.
Without --e2-items a small built-in fallback pool is used (smoke tests only).
"""
from __future__ import annotations

import argparse
import collections
import json
import os
import random
import sys

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, os.path.join(HERE, "..", "e2"))
from aac_vocab import canon  # noqa: E402

SEED = 0
FALLBACK_WRONG = {
    "bedroom": ["television set", "sofa", "bathtub", "kettle"],
    "bathroom": ["bed", "sofa", "refrigerator", "book"],
    "kitchen": ["bed", "pillow", "toilet", "sofa"],
    "dining": ["bed", "toilet", "television set", "blanket"],
    "living room": ["toilet", "bed", "stove", "sink"],
}


def pools(e2_items, bank_rooms):
    """(wrong pool, first-screen pool) per room, from E2's own output."""
    wrong, screen = collections.defaultdict(list), collections.defaultdict(list)
    if e2_items and os.path.exists(e2_items):
        import pandas as pd
        d = pd.read_csv(e2_items)
        d = d[d.method.isin(["clip", "qwen_vl", "blip2_qa", "blip2_caption"])]
        for room, g in d.groupby("room"):
            wrong[room] = sorted({canon(x) for x in g[g.status == "wrong"]["item"].astype(str)})
            screen[room] = sorted({canon(x) for x in g[g["rank"] <= 3]["item"].astype(str)})
    for room in bank_rooms:
        if not wrong.get(room):
            wrong[room] = [canon(x) for x in FALLBACK_WRONG.get(room, FALLBACK_WRONG["bedroom"])]
        if not screen.get(room):
            screen[room] = wrong[room]
    return wrong, screen


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--bank", default=os.path.join(HERE, "intent_bank.json"))
    ap.add_argument("--e2-items", default=None)
    ap.add_argument("--out", default=os.path.join(HERE, "cases.jsonl"))
    a = ap.parse_args()
    bank = json.load(open(a.bank))
    wrong, screen = pools(a.e2_items, {b["room"] for b in bank})
    rng = random.Random(SEED)

    def swap(objs, pool, keep_first):
        """Replace one object (the last, or the only one) with a different item from pool."""
        pool = [p for p in pool if p not in objs]
        if not pool:
            return list(objs), None
        i = len(objs) - 1 if (len(objs) > 1 and keep_first) else rng.randrange(len(objs))
        out = list(objs)
        out[i] = rng.choice(pool)
        return out, i

    n = collections.Counter()
    with open(a.out, "w") as f:
        for b in bank:
            objs = b["objects"]
            cases = {"clean": (objs, None), "partial": (objs[:1], None)}
            cases["wrong_label"] = swap(objs, wrong[b["room"]], keep_first=True)
            cases["wrong_pick"] = swap(objs, screen[b["room"]], keep_first=True)
            for cond, (shown, idx) in cases.items():
                n[cond] += 1
                f.write(json.dumps({"case_id": f"{b['id']}:{cond}", "intent_id": b["id"],
                                    "condition": cond, "room": b["room"], "purpose": b["purpose"],
                                    "objects_true": objs, "objects_shown": shown,
                                    "swapped_index": idx}) + "\n")
    print(f"{sum(n.values())} cases -> {a.out}  {dict(n)}")
    print("pool sizes:", {r: (len(wrong[r]), len(screen[r])) for r in sorted(wrong)})


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/e3/e3_run.py
"""
E3, step 2: every method proposes messages for every case.

    python e3_run.py --bank intent_bank.json --cases cases.jsonl --out e3_outputs \
        --methods template prior embed lm_rank_qwen1.5b lm_gen_qwen1.5b [--limit 8]

Method output is one JSON line per (method, case):
  ranking methods -> "ranked": the intent ids, best first
  generative      -> "text":   the message the model wrote (mapped to intents when scoring)

Resumable: a (method, case) already in the file is skipped.
"""
from __future__ import annotations

import argparse
import collections
import json
import os
import time

MODELS = {                                   # short name -> HF id (all open, non-gated)
    "qwen1.5b": "Qwen/Qwen2.5-1.5B-Instruct",
    "qwen3b": "Qwen/Qwen2.5-3B-Instruct",
    "smol1.7b": "HuggingFaceTB/SmolLM2-1.7B-Instruct",
    "phi3.5": "microsoft/Phi-3.5-mini-instruct",
}
EMBED_ID = "sentence-transformers/all-MiniLM-L6-v2"
SYSTEM = ("You help a person with severe motor impairment who cannot speak. They choose objects "
          "they can see, and you say the sentence they most likely mean, in the first person.")


def user_prompt(case):
    objs = ", ".join(case["objects_shown"])
    return (f"Room: {case['room']}. The person selected, in this order: {objs}.\n"
            "Write the single short sentence they most likely want to say. Reply with the sentence only.")


def rank_prompt(case):
    objs = ", ".join(case["objects_shown"])
    return f"A person in a {case['room']} pointed at: {objs}. They want to say: "


def load_cases(path, limit=None):
    rows = [json.loads(l) for l in open(path)]
    if limit:                                  # dry run: spread over the conditions
        by = collections.defaultdict(list)
        for r in rows:
            by[r["condition"]].append(r)
        rows, i = [], 0
        while len(rows) < min(limit, sum(len(v) for v in by.values())):
            for c in list(by):
                if by[c] and len(rows) < limit:
                    rows.append(by[c].pop(0))
            i += 1
    return rows


# ---------------------------------------------------------------- baselines
def run_template(cases, bank, emit):
    for c in cases:
        objs = c["objects_shown"]
        text = "I want " + (" and ".join(objs) if len(objs) < 3 else ", ".join(objs[:-1]) + " and " + objs[-1]) + "."
        emit("template", c, {"text": text}, 0.0)


def run_prior(cases, bank, emit):
    """No vision, no LM: rank intents by how many of the shown objects they contain, ties
    broken by how common the intent's own objects are elsewhere in the bank (leave-one-out)."""
    freq = collections.Counter(o for b in bank for o in b["objects"])
    pop = {b["id"]: sum(freq[o] - 1 for o in b["objects"]) for b in bank}
    for c in cases:
        shown = set(c["objects_shown"])
        t0 = time.time()
        scored = sorted(bank, key=lambda b: (-len(shown & set(b["objects"])), -pop[b["id"]], b["id"]))
        emit("prior", c, {"ranked": [b["id"] for b in scored[:10]]}, time.time() - t0)


def run_embed(cases, bank, emit, device):
    from sentence_transformers import SentenceTransformer, util
    m = SentenceTransformer(EMBED_ID, device=device)
    texts = [b["utterance"] for b in bank]
    E = m.encode(texts, convert_to_tensor=True, normalize_embeddings=True)
    for c in cases:
        t0 = time.time()
        q = m.encode(f"in the {c['room']}: " + ", ".join(c["objects_shown"]),
                     convert_to_tensor=True, normalize_embeddings=True)
        sims = util.cos_sim(q, E)[0].cpu().numpy()
        order = sims.argsort()[::-1][:10]
        emit("embed", c, {"ranked": [bank[i]["id"] for i in order]}, time.time() - t0)
    del m


def _free():
    import gc
    gc.collect()
    try:
        import torch
        torch.cuda.empty_cache()
    except Exception:
        pass


# ---------------------------------------------------------------- language models
def load_lm(short, device):
    """Built-in implementations only: a model's own downloaded code goes stale against
    newer transformers (Phi-3.5's did, in the cache API)."""
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    mid = MODELS[short]
    tok = AutoTokenizer.from_pretrained(mid)
    dt = torch.float16 if device == "cuda" else torch.float32
    try:
        model = AutoModelForCausalLM.from_pretrained(mid, dtype=dt)      # transformers >= 4.56
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(mid, torch_dtype=dt)
    model = model.to(device).eval()
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    return tok, model


def run_lm_rank(cases, bank, emit, device, short, batch=20):
    """Average token log-likelihood of each candidate message given the objects."""
    import torch
    tok, model = load_lm(short, device)
    cands = [b["utterance"] for b in bank]
    ids = [b["id"] for b in bank]
    for c in cases:
        t0 = time.time()
        prompt = rank_prompt(c)
        p_ids = tok(prompt, return_tensors="pt").input_ids[0]
        scores = []
        for i in range(0, len(cands), batch):
            chunk = cands[i:i + batch]
            seqs = [torch.cat([p_ids, tok(x, return_tensors="pt", add_special_tokens=False).input_ids[0]])
                    for x in chunk]
            n = max(len(s) for s in seqs)
            inp = torch.full((len(seqs), n), tok.pad_token_id)
            att = torch.zeros((len(seqs), n), dtype=torch.long)
            for j, s in enumerate(seqs):
                inp[j, :len(s)], att[j, :len(s)] = s, 1
            inp, att = inp.to(device), att.to(device)
            with torch.no_grad():
                logits = model(input_ids=inp, attention_mask=att, use_cache=False).logits.float().log_softmax(-1)
            for j, s in enumerate(seqs):
                lp = logits[j, torch.arange(len(p_ids) - 1, len(s) - 1), s[len(p_ids):].to(device)]
                scores.append(float(lp.mean()))
        if device == "cuda":
            torch.cuda.synchronize()
        order = sorted(range(len(cands)), key=lambda i: -scores[i])[:10]
        emit(f"lm_rank_{short}", c, {"ranked": [ids[i] for i in order]}, time.time() - t0)
    del model
    _free()


def run_lm_gen(cases, bank, emit, device, short):
    import torch
    tok, model = load_lm(short, device)
    for i, c in enumerate(cases):
        msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user_prompt(c)}]
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        x = tok(text, return_tensors="pt").to(device)
        t0 = time.time()
        with torch.no_grad():
            out = model.generate(**x, max_new_tokens=32, do_sample=False,
                                 pad_token_id=tok.pad_token_id)
        if device == "cuda":
            torch.cuda.synchronize()
        gen = tok.decode(out[0, x["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        gen = gen.split("\n")[0].strip().strip('"')
        if i < 3:
            print(f"  {short} gen: {c['case_id']} {c['objects_shown']} -> {gen!r}", flush=True)
        emit(f"lm_gen_{short}", c, {"text": gen}, time.time() - t0)
    del model
    _free()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--bank", default="intent_bank.json")
    ap.add_argument("--cases", default="cases.jsonl")
    ap.add_argument("--out", default="e3_outputs")
    ap.add_argument("--methods", nargs="+", default=["template", "prior", "embed"])
    ap.add_argument("--limit", type=int, default=None)
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    bank = json.load(open(a.bank))
    cases = load_cases(a.cases, a.limit)
    path = os.path.join(a.out, "outputs.dry.jsonl" if a.limit else "outputs.jsonl")

    done = set()
    if os.path.exists(path):                     # drop a half-written last line
        good = []
        for l in open(path):
            try:
                o = json.loads(l)
                done.add((o["method"], o["case_id"]))
                good.append(l if l.endswith("\n") else l + "\n")
            except json.JSONDecodeError:
                pass
        open(path, "w").writelines(good)
    try:
        import torch
        device = "cuda" if torch.cuda.is_available() else "cpu"
    except ImportError:
        device = "cpu"
    print(f"{len(cases)} cases | {len(bank)} intents | device {device} | methods {a.methods}", flush=True)

    f = open(path, "a")
    n = collections.Counter()

    def emit(method, case, payload, secs):
        f.write(json.dumps({"method": method, "case_id": case["case_id"], "intent_id": case["intent_id"],
                            "condition": case["condition"], "seconds": round(secs, 4), **payload}) + "\n")
        f.flush()
        n[method] += 1
        if n[method] % 50 == 0:
            print(f"  {method}: {n[method]} cases", flush=True)

    for m in a.methods:
        todo = [c for c in cases if (m, c["case_id"]) not in done]
        if not todo:
            print(f"{m}: already done")
            continue
        print(f"{m}: {len(todo)} cases ...", flush=True)
        t0 = time.time()
        try:
            if m == "template":
                run_template(todo, bank, emit)
            elif m == "prior":
                run_prior(todo, bank, emit)
            elif m == "embed":
                run_embed(todo, bank, emit, device)
            elif m.startswith("lm_rank_"):
                run_lm_rank(todo, bank, emit, device, m[len("lm_rank_"):])
            elif m.startswith("lm_gen_"):
                run_lm_gen(todo, bank, emit, device, m[len("lm_gen_"):])
            else:
                raise SystemExit(f"unknown method {m}")
        except Exception as e:                      # keep going: the rest of the run is still useful
            import traceback
            traceback.print_exc()
            print(f"!! {m} FAILED ({type(e).__name__}: {e}); continuing with the other methods", flush=True)
            _free()
            continue
        print(f"{m}: done in {time.time() - t0:.0f} s", flush=True)
    f.close()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/e3/e3_score.py
"""
E3, step 3: score every method.

    python e3_score.py --bank intent_bank.json --cases cases.jsonl \
        --outputs e3_outputs/outputs.jsonl --out e3_results

Ranking methods are scored directly. Generative methods are mapped to the intent bank by
sentence embedding (the generated message is matched against each intent's utterance and its
two paraphrases, best of the three), which is the automatic score; the blinded human sheet
covers the same generations.

Outputs: e3_summary.csv, e3_by_condition.csv, e3_paired_differences.csv, e3_per_case.csv,
e3_generations.csv (for the human check), E3_REPORT.md.
"""
from __future__ import annotations

import argparse
import json
import os
import re

import numpy as np
import pandas as pd

SEED, N_BOOT = 0, 5000
CONDITIONS = ["clean", "partial", "wrong_label", "wrong_pick"]
EMBED_ID = "sentence-transformers/all-MiniLM-L6-v2"


def boot(v, rng):
    v = np.asarray(v, float)
    v = v[np.isfinite(v)]
    if len(v) < 2:
        return (np.nan, np.nan)
    b = v[rng.integers(0, len(v), (N_BOOT, len(v)))].mean(1)
    return round(float(np.percentile(b, 2.5)), 4), round(float(np.percentile(b, 97.5)), 4)


def paired(a, b, rng):
    d = np.asarray(a, float) - np.asarray(b, float)
    if len(d) < 2:
        return np.nan, (np.nan, np.nan)
    m = d[rng.integers(0, len(d), (N_BOOT, len(d)))].mean(1)
    return round(float(d.mean()), 4), (round(float(np.percentile(m, 2.5)), 4),
                                       round(float(np.percentile(m, 97.5)), 4))


def _tok(s):
    return set(re.findall(r"[a-z]+", str(s).lower())) - {"the", "a", "an", "i", "my", "me", "to",
                                                         "please", "it", "is", "of", "and", "you"}


class Mapper:
    """Generated text -> ranked intent ids. Sentence embeddings, with a token-overlap
    fallback so the smoke tests run without downloading a model."""

    def __init__(self, bank):
        self.bank = bank
        self.texts = [[b["utterance"]] + b["paraphrases"] for b in bank]
        self.model = None
        try:
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer(EMBED_ID)
            flat = [t for group in self.texts for t in group]
            self.E = self.model.encode(flat, convert_to_tensor=True, normalize_embeddings=True)
            self.owner = [i for i, group in enumerate(self.texts) for _ in group]
            print("mapper: sentence embeddings")
        except Exception as e:                                   # offline / no package
            print(f"mapper: token overlap fallback ({type(e).__name__})")

    def rank(self, text, k=10):
        if self.model is not None:
            import torch
            q = self.model.encode(text, convert_to_tensor=True, normalize_embeddings=True)
            sims = (self.E @ q).cpu().numpy()
            best = {}
            for s, i in zip(sims, self.owner):
                best[i] = max(best.get(i, -9), float(s))
        else:
            t = _tok(text)
            best = {i: max(len(t & _tok(x)) / max(1, len(t | _tok(x))) for x in group)
                    for i, group in enumerate(self.texts)}
        order = sorted(best, key=lambda i: (-best[i], self.bank[i]["id"]))[:k]
        return [self.bank[i]["id"] for i in order]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--bank", default="intent_bank.json")
    ap.add_argument("--cases", default="cases.jsonl")
    ap.add_argument("--outputs", default="e3_outputs/outputs.jsonl")
    ap.add_argument("--out", default="e3_results")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    rng = np.random.default_rng(SEED)

    bank = json.load(open(a.bank))
    byid = {b["id"]: b for b in bank}
    cases = {c["case_id"]: c for c in (json.loads(l) for l in open(a.cases))}
    outs = []
    for l in open(a.outputs):
        try:
            outs.append(json.loads(l))
        except json.JSONDecodeError:
            pass
    last = {(o["method"], o["case_id"]): o for o in outs if o["case_id"] in cases}
    methods = sorted({m for m, _ in last})
    common = set.intersection(*[{c for (m, c) in last if m == meth} for meth in methods])
    print("cases per method:", {m: sum(1 for (mm, _) in last if mm == m) for m in methods},
          "| scored (common):", len(common), flush=True)

    mapper = Mapper(bank) if any("text" in o for o in last.values()) else None
    rows, gens = [], []
    for (meth, cid), o in sorted(last.items()):
        if cid not in common:
            continue
        c = cases[cid]
        ranked = o.get("ranked")
        if ranked is None:
            ranked = mapper.rank(o.get("text", ""))
            gens.append({"method": meth, "case_id": cid, "intent_id": c["intent_id"],
                         "condition": c["condition"], "objects_shown": ", ".join(c["objects_shown"]),
                         "generated": o.get("text", ""), "reference": byid[c["intent_id"]]["utterance"],
                         "mapped_to": byid[ranked[0]]["utterance"] if ranked else "",
                         "auto_correct": bool(ranked and ranked[0] == c["intent_id"])})
        gold = c["intent_id"]
        rows.append({"method": meth, "case_id": cid, "intent_id": gold, "condition": c["condition"],
                     "purpose": c["purpose"], "room": c["room"], "seconds": o.get("seconds", np.nan),
                     "top1": float(bool(ranked[:1] == [gold])), "top3": float(gold in ranked[:3]),
                     "top5": float(gold in ranked[:5]),
                     "rank": (ranked.index(gold) + 1) if gold in ranked else np.nan})
    per = pd.DataFrame(rows)
    per.to_csv(os.path.join(a.out, "e3_per_case.csv"), index=False)
    if gens:
        pd.DataFrame(gens).to_csv(os.path.join(a.out, "e3_generations.csv"), index=False)

    summ = []
    for meth, d in per.groupby("method"):
        r = {"method": meth, "cases": len(d), "median_seconds": float(np.nanmedian(d.seconds))}
        for k in ("top1", "top3", "top5"):
            r[k] = round(float(d[k].mean()), 4)
            r[k + "_ci"] = boot(d[k].values, rng)
        for cond in CONDITIONS:
            dc = d[d.condition == cond]
            if len(dc):
                r[f"top1_{cond}"] = round(float(dc.top1.mean()), 4)
                r[f"top3_{cond}"] = round(float(dc.top3.mean()), 4)
        for p in ("needs", "information", "social", "etiquette"):
            dp = d[d.purpose == p]
            if len(dp):
                r[f"top3_{p}"] = round(float(dp.top3.mean()), 4)
        summ.append(r)
    summ = pd.DataFrame(summ).sort_values("top3", ascending=False)
    summ.to_csv(os.path.join(a.out, "e3_summary.csv"), index=False)

    by_cond = per.groupby(["method", "condition"]).agg(n=("case_id", "count"), top1=("top1", "mean"),
                                                       top3=("top3", "mean")).reset_index()
    by_cond.to_csv(os.path.join(a.out, "e3_by_condition.csv"), index=False)

    # paired differences against the retrieval baseline and the template floor, per condition
    diffs = []
    wide = {k: per.pivot_table(index="case_id", columns="method", values=k) for k in ("top1", "top3")}
    base = [b for b in ("embed", "template", "prior") if b in set(per.method)]
    for x in sorted(set(per.method)):
        for y in base:
            if x == y:
                continue
            for k, w in wide.items():
                mean, ci = paired(w[x].values, w[y].values, rng)
                diffs.append({"a": x, "b": y, "metric": k, "a_minus_b": mean,
                              "ci_lo": ci[0], "ci_hi": ci[1], "n": len(w)})
                for cond in CONDITIONS:
                    ids = per[per.condition == cond].case_id.unique()
                    wc = w.loc[w.index.intersection(ids)]
                    mean, ci = paired(wc[x].values, wc[y].values, rng)
                    diffs.append({"a": x, "b": y, "metric": f"{k}_{cond}", "a_minus_b": mean,
                                  "ci_lo": ci[0], "ci_hi": ci[1], "n": len(wc)})
    diffs = pd.DataFrame(diffs, columns=["a", "b", "metric", "a_minus_b", "ci_lo", "ci_hi", "n"])
    diffs.to_csv(os.path.join(a.out, "e3_paired_differences.csv"), index=False)

    L = ["# E3: from selected objects to the intended message", "",
         f"{per.case_id.nunique()} cases ({len(bank)} intents x {len(CONDITIONS)} conditions), "
         f"methods: {', '.join(sorted(set(per.method)))}.", "",
         "## Summary (top-1 = offered first, top-3 = on the first screen of candidate messages)", "",
         summ.to_markdown(index=False), "", "## By condition", "", by_cond.to_markdown(index=False), "",
         "## Paired differences vs the baselines", "",
         diffs[diffs.metric.isin(["top1", "top3"])].to_markdown(index=False), ""]
    open(os.path.join(a.out, "E3_REPORT.md"), "w").write("\n".join(L) + "\n")
    print(summ.to_string(index=False))
    print("\nwrote", a.out)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/e3/smoke_test_e3.py
"""
E3 smoke test: runs without a GPU, without downloading any model.

    python smoke_test_e3.py

Checks the bank, the case builder (determinism and what each condition changes), the two
cheap methods, an oracle and an anti-oracle through the scorer, resume after a half-written
line, and that the report is written.
"""
import json
import os
import shutil
import subprocess
import sys
import tempfile

HERE = os.path.dirname(os.path.abspath(__file__))
PY = sys.executable
ok = fail = 0


def check(cond, msg):
    global ok, fail
    print(("OK   " if cond else "FAIL ") + msg)
    ok, fail = ok + bool(cond), fail + (not cond)


def run(*args):
    r = subprocess.run([PY] + list(args), capture_output=True, text=True, cwd=HERE)
    if r.returncode:
        print(r.stdout[-2000:], r.stderr[-2000:])
    return r


tmp = tempfile.mkdtemp()
try:
    # ---- bank
    r = run("build_intent_bank.py")
    check(r.returncode == 0, "intent bank builds and every object is in the AAC vocabulary")
    bank = json.load(open(os.path.join(HERE, "intent_bank.json")))
    check(len(bank) == 60, f"60 intents ({len(bank)})")
    check(len({b["utterance"] for b in bank}) == 60, "no duplicate utterances")
    check(all(1 <= len(b["objects"]) <= 3 for b in bank), "1-3 objects per intent")
    check(all(len(b["paraphrases"]) == 2 for b in bank), "two paraphrases per intent")

    # ---- cases
    c1, c2 = os.path.join(tmp, "c1.jsonl"), os.path.join(tmp, "c2.jsonl")
    run("e3_build_cases.py", "--out", c1)
    run("e3_build_cases.py", "--out", c2)
    check(open(c1).read() == open(c2).read(), "case building is deterministic")
    cases = [json.loads(l) for l in open(c1)]
    check(len(cases) == 240, f"240 cases ({len(cases)})")
    byc = {}
    for c in cases:
        byc.setdefault(c["condition"], []).append(c)
    check(all(len(c["objects_shown"]) == 1 for c in byc["partial"]), "partial shows one object")
    check(all(c["objects_shown"] == c["objects_true"] for c in byc["clean"]), "clean is unchanged")
    for cond in ("wrong_label", "wrong_pick"):
        changed = [c for c in byc[cond] if c["objects_shown"] != c["objects_true"]]
        check(len(changed) == 60, f"{cond} changes every case ({len(changed)}/60)")
        check(all(len(c["objects_shown"]) == len(c["objects_true"]) for c in byc[cond]),
              f"{cond} keeps the number of selections")
        multi = [c for c in byc[cond] if len(c["objects_true"]) > 1]
        check(all(c["objects_shown"][0] == c["objects_true"][0] for c in multi),
              f"{cond} keeps the first selection when there are several")

    # ---- cheap methods
    out = os.path.join(tmp, "out")
    r = run("e3_run.py", "--cases", c1, "--out", out, "--methods", "template", "prior")
    check(r.returncode == 0, "template and prior run")
    lines = open(os.path.join(out, "outputs.jsonl")).read().splitlines()
    check(len(lines) == 480, f"one line per method and case ({len(lines)})")

    # resume after a half-written last line
    with open(os.path.join(out, "outputs.jsonl"), "a") as f:
        f.write('{"method": "prior", "case_id": "I01:cle')
    r = run("e3_run.py", "--cases", c1, "--out", out, "--methods", "template", "prior")
    lines2 = open(os.path.join(out, "outputs.jsonl")).read().splitlines()
    check(r.returncode == 0 and len(lines2) == 480, f"resume drops the broken line and re-runs nothing ({len(lines2)})")

    # dry run writes a separate file
    r = run("e3_run.py", "--cases", c1, "--out", out, "--methods", "template", "--limit", "8")
    check(os.path.exists(os.path.join(out, "outputs.dry.jsonl")) and len(lines2) == 480,
          "a dry run cannot overwrite the full output")

    # ---- scorer: oracle and anti-oracle
    ids = [b["id"] for b in bank]
    orc = os.path.join(tmp, "oracle.jsonl")
    with open(orc, "w") as f:
        for c in cases:
            gold = c["intent_id"]
            others = [i for i in ids if i != gold]
            f.write(json.dumps({"method": "oracle", "case_id": c["case_id"], "seconds": 0,
                                "ranked": [gold] + others[:9]}) + "\n")
            f.write(json.dumps({"method": "anti", "case_id": c["case_id"], "seconds": 0,
                                "ranked": others[:10]}) + "\n")
            f.write(json.dumps({"method": "third", "case_id": c["case_id"], "seconds": 0,
                                "ranked": others[:2] + [gold] + others[2:9]}) + "\n")
            f.write(json.dumps({"method": "gen_gold", "case_id": c["case_id"], "seconds": 0,
                                "text": [b for b in bank if b["id"] == gold][0]["utterance"]}) + "\n")
    res = os.path.join(tmp, "res")
    r = run("e3_score.py", "--cases", c1, "--outputs", orc, "--out", res)
    check(r.returncode == 0, "scorer runs")
    import pandas as pd
    s = pd.read_csv(os.path.join(res, "e3_summary.csv")).set_index("method")
    check(s.loc["oracle", "top1"] == 1.0 and s.loc["oracle", "top3"] == 1.0, "oracle scores 1.0")
    check(s.loc["anti", "top1"] == 0.0 and s.loc["anti", "top3"] == 0.0, "anti-oracle scores 0.0")
    check(s.loc["third", "top1"] == 0.0 and s.loc["third", "top3"] == 1.0, "gold at rank 3 gives top1 0, top3 1")
    check(s.loc["gen_gold", "top1"] > 0.9, f"a generation equal to the reference maps back to it "
                                           f"({s.loc['gen_gold', 'top1']:.2f}, token fallback)")
    gen = pd.read_csv(os.path.join(res, "e3_generations.csv"))
    check(len(gen) == 240 and "reference" in gen.columns, "generations sheet written for the human check")
    check(os.path.exists(os.path.join(res, "E3_REPORT.md")), "report written")

    # unequal coverage: only the common cases are scored
    part = os.path.join(tmp, "part.jsonl")
    with open(part, "w") as f:
        for i, l in enumerate(open(orc)):
            o = json.loads(l)
            if o["method"] == "anti" and i % 3 == 0:
                continue
            f.write(l)
    r = run("e3_score.py", "--cases", c1, "--outputs", part, "--out", os.path.join(tmp, "res2"))
    s2 = pd.read_csv(os.path.join(tmp, "res2", "e3_summary.csv"))
    check(s2.cases.nunique() == 1, f"every method scored on the same cases ({sorted(set(s2.cases))})")
finally:
    shutil.rmtree(tmp, ignore_errors=True)

print(f"\n{ok} passed, {fail} failed")
sys.exit(1 if fail else 0)


## 1. Build the intent bank and the cases (a few seconds)
Check: **60 intents**, **240 cases**, and the E2 pool sizes (if E2's items file was not found it falls back to a small built-in pool — note it if that happens, because the fallback pool changes the result).

In [ ]:
%cd /content/e3
!python build_intent_bank.py
import os
ITEMS = f'{E2}/results_v2/e2_items.csv' if os.path.exists(f'{E2}/results_v2/e2_items.csv') else f'{E2}/results/e2_items.csv'
!python e3_build_cases.py --e2-items "{ITEMS}" --out "{WORK}/cases.jsonl"

## 2. Self-test (about 1 minute)
Every line must say OK, and the last line must say **0 failed**.

In [ ]:
!cd /content/e3 && python smoke_test_e3.py

## 3. Baselines (about 2 minutes)

In [ ]:
!cd /content/e3 && python e3_run.py --bank intent_bank.json --cases "{WORK}/cases.jsonl" --out "{WORK}/outputs" --methods template prior embed

## 4. Dry run: the smallest model on 8 cases (about 3 minutes, mostly the download)
Check that the generated sentences look like real sentences. **If they are empty or nonsense, stop and read this output before going on.**

In [ ]:
!cd /content/e3 && python e3_run.py --bank intent_bank.json --cases "{WORK}/cases.jsonl" --out /content/e3_dry --limit 8 --methods lm_rank_qwen1.5b lm_gen_qwen1.5b

## 5. The four language models (about 60–80 minutes; resumable)
Each model is downloaded, used for ranking and for writing, then freed. If Colab disconnects, just run this cell again.

In [ ]:
!cd /content/e3 && python e3_run.py --bank intent_bank.json --cases "{WORK}/cases.jsonl" --out "{WORK}/outputs" --methods lm_rank_qwen1.5b lm_gen_qwen1.5b lm_rank_smol1.7b lm_gen_smol1.7b lm_rank_qwen3b lm_gen_qwen3b lm_rank_phi3.5 lm_gen_phi3.5

## 6. Score and report (1 minute)

In [ ]:
!cd /content/e3 && python e3_score.py --bank intent_bank.json --cases "{WORK}/cases.jsonl" --outputs "{WORK}/outputs/outputs.jsonl" --out "{WORK}/results"

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f'{WORK}/results/E3_REPORT.md').read()))

## 7. Download
Download `E3_results.zip`.

In [ ]:
import shutil, os
shutil.rmtree('/content/E3_results', ignore_errors=True); os.makedirs('/content/E3_results')
shutil.copytree(f'{WORK}/results', '/content/E3_results/results', dirs_exist_ok=True)
for f in ['cases.jsonl', 'outputs/outputs.jsonl']:
    shutil.copy(f'{WORK}/{f}', '/content/E3_results/' + os.path.basename(f))
shutil.copy('/content/e3/intent_bank.json', '/content/E3_results/intent_bank.json')
shutil.make_archive('/content/E3_results', 'zip', '/content/E3_results')
from google.colab import files; files.download('/content/E3_results.zip')